# Ordered Logistic Regression Results for Adoption Predictors of Indigenous and Modern Knowledge in Rangeland Management Practices, Northern Kenya Exploration with `mlcroissant`
This notebook provides a step-by-step guide for loading and exploring the **FAIR^2 Dataset** using the [`mlcroissant`](https://github.com/mlcommons/croissant) library, referencing all entities by their `@id`.

### Dataset Source
The dataset source is provided via a Croissant schema URL:

`https://sen.science/doi/10.71728/senscience.y7m0-f273/fair2.json`

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install -U mlcroissant

## 1. Data Loading

Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json

# Define the Croissant schema URL
croissant_url = "https://sen.science/doi/10.71728/senscience.y7m0-f273/fair2.json"

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)

meta = dataset.metadata
print(f"Dataset Title: {meta.name}\n\nDescription: {meta.description}\n")
print(f"Identifier: {meta.identifier}\nVersion: {meta.version}\nLicense: {meta.license}\n")

## 2. Data Overview

Review available **record sets** (`cr:RecordSet`), their fields (`cr:Field`), and all entity `@id`s in the dataset.

In [ ]:
# List all record sets and their field @id's
record_sets = dataset.record_sets
print(f"Total Record Sets: {len(record_sets)}\n")
overview = []
for rset in record_sets:
    print(f"Record Set: {rset.metadata['@id']} (name: {rset.name})")
    fields = rset.fields
    field_ids = [f.metadata['@id'] for f in fields]
    print(f"  Fields (by @id): {field_ids}")
    overview.append({
        'record_set_id': rset.metadata['@id'],
        'name': rset.name,
        'fields': field_ids
    })

# If there are no record sets, print a message
if not record_sets:
    print("No record sets declared in the Croissant file. If this is unexpected, please check the schema or contact the data provider.")

## 3. Data Extraction

Load data from a specific record set into a DataFrame. Use the record set and field `@id`s from the overview above.

In [ ]:
# Extract data from one or more record sets (referenced by @id)

# Collect available record set @ids
all_record_set_ids = [rs.metadata['@id'] for rs in dataset.record_sets]
dataframes = {}

if all_record_set_ids:
    for rs_id in all_record_set_ids:
        records = list(dataset.records(record_set=rs_id))
        dataframes[rs_id] = pd.DataFrame(records)
else:
    print("No record sets available, so data extraction cannot be demonstrated.")

# Display columns of the first dataframe extracted (choose the first available record set @id)
if dataframes:
    first_rs = next(iter(dataframes.keys()))
    print(f"Columns for record set {first_rs}: {dataframes[first_rs].columns.tolist()}")
    display(dataframes[first_rs].head())
else:
    print("No dataframes loaded.")

## 4. Exploratory Data Analysis (EDA)

Process the data: filter, normalize, and group data using field `@id`s for referencing. This demonstrates filtering records by a numeric field, normalizing values, and aggregating by another field.

In [ ]:
# Pick a numeric field @id to analyze from the first available record set
import numpy as np

if dataframes:
    record_set_id = first_rs
    df = dataframes[record_set_id]
    # Attempt to automatically find a numeric field by field @id
    numeric_field_id = None
    for col in df.columns:
        if np.issubdtype(df[col].dtype, np.number):
            numeric_field_id = col
            break
    if numeric_field_id is not None:
        threshold = df[numeric_field_id].median() if not df[numeric_field_id].isnull().all() else 0
        filtered_df = df[df[numeric_field_id] > threshold]
        print(f"Filtered records with {numeric_field_id} > {threshold}:")
        display(filtered_df.head())

        filtered_df[f"{numeric_field_id}_normalized"] = (
            (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) /
            filtered_df[numeric_field_id].std()
        )
        print(f"Normalized {numeric_field_id} for filtered records:")
        display(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

        # Try to pick a group field (categorical) by @id
        group_field_id = None
        for col in df.columns:
            if df[col].dtype == object and col != numeric_field_id:
                group_field_id = col
                break
        if group_field_id:
            grouped_df = filtered_df.groupby(group_field_id).mean(numeric_only=True)
            print(f"Grouped data by {group_field_id}:")
            display(grouped_df.head())
        else:
            print("No suitable group field found.")
    else:
        print("No numeric field available for EDA in this record set.")
else:
    print("No data available for EDA.")

## 5. Visualization

Visualize the distribution of a numeric field and illustrate group comparisons using field `@id`s.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if dataframes and numeric_field_id is not None:
    plt.figure(figsize=(8, 4))
    sns.histplot(df[numeric_field_id].dropna(), kde=True, color="skyblue")
    plt.title(f"Distribution of {numeric_field_id}")
    plt.xlabel(numeric_field_id)
    plt.ylabel("Frequency")
    plt.show()
    if group_field_id:
        plt.figure(figsize=(8, 4))
        sns.boxplot(data=df, x=group_field_id, y=numeric_field_id)
        plt.title(f"Boxplot of {numeric_field_id} by {group_field_id}")
        plt.xlabel(group_field_id)
        plt.ylabel(numeric_field_id)
        plt.xticks(rotation=45)
        plt.show()
else:
    print("No numeric field found for visualization.")

## 6. Conclusion

This notebook demonstrated how to use the `mlcroissant` library to:  
- Load a Croissant-described dataset and review its metadata (referenced by `@id`),  
- Enumerate record sets and fields using their `@id`,  
- Extract tabular data from record sets into DataFrames,  
- Perform filtering, normalization, and basic aggregation using field and record set `@id`s,  
- Visualize distributions and group differences.  

For further analysis or research, continue structured EDA using the field and record set `@id` approach to ensure reproducibility and schema-aligned processing. For more information, visit [mlcroissant documentation](https://mlcommons.github.io/croissant/python/reference/).